In [2]:
#Checking if API Works and hence LLM Created

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=api_key,
    temperature=0.2
)

response = llm.invoke("Write a short poem on my love for dosa")
print(response.content)


[{'type': 'text', 'text': 'Golden-crisp and paper-thin,  \nThat’s where all my cravings win.  \nSizzling softly on the pan,  \nThe truest love since time began.  \n\nFolded warm with fragrant ghee,  \nA savory dream of ecstasy.  \nDipped in sambar, chutney white,  \nMy heart surrenders to every bite.', 'extras': {'signature': 'EvIPCu8PAWkUfRPaCEa2Ge3YySUK+Dcjf+Q24Yu+6dte0Qs2TNRroZNXpyttUimKEi4cHx2UVZaxfnEWXVLxKGT+qgj+U1Ibku5080v0uvxK5OVkpQaa+2WuOqZVBtnBTI3ya1CyhoAu8nJLjhnn4+edSP6PGQshQb3rZKfTw6/Jy/tRkqTnrRb8LM/y9/2M2nWCGwUX4M+p37KmzHFOF551fhoAm1stN0luXW14kBDCtZIjSEaEiEy5E+SS7L7M/iSHDcRSFoUnAI5rXB633zI4qiBsijB4GCNjQEyHjk2sO+CPuLvq9fojnTcaNbSdbU2uKaxLDdXli85nF5DcEqAc8ZNavH+zRu707HyAaMBWAUXhqySrFw4PSRQUWQOhJv3YitsIqTKvO/sFVIXnUUbBOdN0zcvSbcytyCCCfNF4azKajUTD0LY86yh9kQf9JP81ullMLWspQozoDiti0F5Bu6IM849wCPeVppN3UiUGcQzPdSGoQ+nF3AxgSaC4TyH+10tPuLPPwJiCPJZg4huHRrw0pgBx9f0AcmhSfMwkBY8y1v+zSNLlHsGEokvWSzJz4vPVPdftW3FebfrloVQgzE8epBs2vuWv1RTIhMDuEhJ8UAN67t/89E6DyfdQn9ZCSoeoTpCC/i7EqhCpWwALpErVJ55

In [5]:
#Connection to SQL Workbench Databade

from langchain_community.utilities import SQLDatabase

db_user = "root"
db_password = "root"
db_host = "localhost"
db_name = "atliq_tshirts"

db = SQLDatabase.from_uri(
    f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",
    sample_rows_in_table_info=4
)

print(db.get_usable_table_names())
print(db.table_info)

['discounts', 't_shirts']

CREATE TABLE discounts (
	discount_id INTEGER NOT NULL AUTO_INCREMENT, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount DECIMAL(5, 2), 
	PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), 
	CONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
4 rows from discounts table:
discount_id	t_shirt_id	pct_discount
1	1	10.00
2	2	15.00
3	3	20.00
4	4	5.00
*/


CREATE TABLE t_shirts (
	t_shirt_id INTEGER NOT NULL AUTO_INCREMENT, 
	brand ENUM('Van Huesen','Levi','Nike','Adidas') NOT NULL, 
	color ENUM('Red','Blue','Black','White') NOT NULL, 
	size ENUM('XS','S','M','L','XL') NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_chk_1 CHECK ((`price` between 10 and 50))
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
4 rows from t_shirts table:
t_sh

In [6]:
from langchain_experimental.sql import SQLDatabaseChain

db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)
qns1 = db_chain.invoke({"query": "How many t-shirts do we have left for Nike in extra small size and white color?"})
print(qns1)



> Entering new SQLDatabaseChain chain...
How many t-shirts do we have left for Nike in extra small size and white color?
SQLQuery:SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Nike' AND `color` = 'White' AND `size` = 'XS';
SQLResult: [(Decimal('53'),)]
Answer:There are 53 Nike t-shirts left in extra small size and white color.
> Finished chain.
{'query': 'How many t-shirts do we have left for Nike in extra small size and white color?', 'result': 'There are 53 Nike t-shirts left in extra small size and white color.'}


In [ ]:
from langchain_experimental.sql import SQLDatabaseChain

db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)
qns2 = db_chain.invoke({"query": "What is the total number of states in nepal?"})
print(qns2)